# Fertilizer Recommendation Model Training

This notebook implements a fertilizer recommendation system using a Random Forest Classifier based on soil and environmental features.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

## 1. Load Dataset

In [ ]:
# Load the dataset
dataset_path = 'dataset/Fertilizer Prediction (1).csv'
df = pd.read_csv(dataset_path)

# Display first few rows
df.head()

## 2. Preprocessing

We need to encode the categorical features ('Soil Type' and 'Crop Type') and the target variable ('Fertilizer Name').

In [ ]:
# Check for null values
print("Missing values:\n", df.isnull().sum())

# Encoding categorical columns
le_soil = LabelEncoder()
df['Soil Type'] = le_soil.fit_transform(df['Soil Type'])

le_crop = LabelEncoder()
df['Crop Type'] = le_crop.fit_transform(df['Crop Type'])

le_fertilizer = LabelEncoder()
df['Fertilizer Name'] = le_fertilizer.fit_transform(df['Fertilizer Name'])

print("\nEncoded Soil Types:", dict(zip(le_soil.classes_, le_soil.transform(le_soil.classes_))))
print("Encoded Crop Types:", dict(zip(le_crop.classes_, le_crop.transform(le_crop.classes_))))
print("Encoded Fertilizer Names:", dict(zip(le_fertilizer.classes_, le_fertilizer.transform(le_fertilizer.classes_))))

## 3. Data Splitting

Split the data into training (80%) and testing (20%) sets.

In [ ]:
X = df.drop('Fertilizer Name', axis=1)
y = df['Fertilizer Name']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)

## 4. Model Training

Using a Random Forest Classifier as recommended in the documentation.

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

print("Model training complete.")

## 5. Evaluation

In [ ]:
y_pred = rf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy Score: {accuracy * 100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=le_fertilizer.classes_))

## 6. Visualization

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le_fertilizer.classes_, yticklabels=le_fertilizer.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

# Feature Importance
feature_importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(x=feature_importance, y=feature_importance.index)
plt.title('Feature Importance')
plt.show()

## 7. Save Model & Encoders

In [ ]:
joblib.dump(rf, 'fertilizer_rf_model.pkl')
joblib.dump(le_soil, 'soil_encoder.pkl')
joblib.dump(le_crop, 'crop_encoder.pkl')
joblib.dump(le_fertilizer, 'fertilizer_encoder.pkl')

print("Model and encoders saved successully.")